In [ ]:
#!/usr/bin/env python3
"""
Aggregate the per-combination gold-standard comparison CSVs produced by
evaluate_model.py into overall statistics and summary figures.

Reads:  results/<student>/<day>/<behavior>/gold_comparison_<model>.csv
Writes: figures/model_comparison_report/*.png

Usage:
  python3 generate_comparison_report.py
"""
import csv
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path("/Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT")
RESULTS_ROOT = PROJECT_ROOT / "results"
OUT_DIR = PROJECT_ROOT / "figures" / "model_comparison_report"

STUDENTS  = ["Taylor Swift", "DaPaw", "Rose", "SJ3747"]
DAYS      = ["day 1", "day 2"]
BEHAVIORS = ["enacting", "planning", "reflecting", "monitoring", "interacting"]
STUDENT_SLUGS = {"Taylor Swift": "taylor", "DaPaw": "dapaw", "Rose": "rose", "SJ3747": "sj3747"}
DAY_TAGS = {"day 1": "day1", "day 2": "day2"}

MODEL_TAGS  = ["sonnet4_6", "gpt5_2", "gpt4o"]
MODEL_NAMES = {"sonnet4_6": "Sonnet 4.6", "gpt5_2": "GPT-5.2", "gpt4o": "GPT-4o"}
MODEL_COLORS = {"Sonnet 4.6": "#2a78d6", "GPT-5.2": "#1baf7a", "GPT-4o": "#eda100"}

CAT_LABELS = {
    1: "1 Exact Equal", 2: "2 Start<Gold/End=", 3: "3 Start=/End>Gold",
    4: "4 Start</End inside", 5: "5 Start inside/End>", 6: "6 Start=/End<Gold",
    7: "7 Start>Gold/End=", 8: "8 AI inside Gold", 9: "9 AI contains Gold",
    10: "10 Complete Miss",
}

## Data collection

In [ ]:
def collect_comparisons():
    """Read every gold_comparison_<model>.csv and return:
    - agg: category counts per model, summed across all combinations
    - by_behavior / by_student: exact/miss counts per model, split by slice
    - per_combo: category counts per model, keyed by combination name
    """
    agg = {m: {c: 0 for c in range(1, 11)} for m in MODEL_TAGS}
    by_behavior = {}
    by_student = {}
    per_combo = {}

    for student in STUDENTS:
        stu = STUDENT_SLUGS[student]
        for day in DAYS:
            day_tag = DAY_TAGS[day]
            for beh in BEHAVIORS:
                combo_dir = RESULTS_ROOT / stu / day_tag / beh
                if not combo_dir.exists():
                    continue
                combo_data = {}
                for model_tag in MODEL_TAGS:
                    csv_path = combo_dir / f"gold_comparison_{model_tag}.csv"
                    if not csv_path.exists():
                        continue
                    cat_counts = {c: 0 for c in range(1, 11)}
                    with open(csv_path, newline="", encoding="utf-8") as f:
                        for row in csv.DictReader(f):
                            cat = int(row["category"])
                            cat_counts[cat] += 1
                            agg[model_tag][cat] += 1

                            beh_bucket = by_behavior.setdefault(beh, {}).setdefault(
                                model_tag, {"exact": 0, "miss": 0, "total": 0})
                            beh_bucket["total"] += 1
                            beh_bucket["exact"] += cat == 1
                            beh_bucket["miss"] += cat == 10

                            stu_bucket = by_student.setdefault(student, {}).setdefault(
                                model_tag, {"exact": 0, "miss": 0, "total": 0})
                            stu_bucket["total"] += 1
                            stu_bucket["exact"] += cat == 1
                            stu_bucket["miss"] += cat == 10
                    combo_data[model_tag] = cat_counts
                if combo_data:
                    per_combo[f"{student}/{day}/{beh}"] = combo_data

    return agg, by_behavior, by_student, per_combo


def count_combo_wins(per_combo):
    wins = {m: 0 for m in MODEL_TAGS}
    ties = 0
    for combo_data in per_combo.values():
        rates = {}
        for model_tag, cats in combo_data.items():
            total = sum(cats.values())
            rates[model_tag] = cats.get(1, 0) / total * 100 if total else 0
        best = max(rates.values())
        winners = [m for m, r in rates.items() if abs(r - best) < 1e-9]
        if len(winners) == 1:
            wins[winners[0]] += 1
        else:
            ties += 1
    return wins, ties


def rate_pct(bucket, field):
    return bucket[field] / bucket["total"] * 100 if bucket["total"] else 0.0

## Chart rendering

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.edgecolor": "#c9c8c0",
    "axes.labelcolor": "#0b0b0b",
    "text.color": "#0b0b0b",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})


def _style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.grid(axis="x", color="#e4e2dc", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)


def grouped_hbar_chart(data, title, subtitle, out_path, xlim, xlabel="%"):
    """data: {group_label: {model_name: value}}"""
    models = list(MODEL_COLORS.keys())
    groups = list(data.keys())
    fig_h = max(2.2, 0.9 * len(groups) + 1.0)
    fig, ax = plt.subplots(figsize=(8, fig_h), dpi=200)
    bar_h, gap = 0.24, 0.03
    y_base = np.arange(len(groups))[::-1]

    for i, model in enumerate(models):
        offset = (i - 1) * (bar_h + gap)
        ys = y_base + offset
        vals = [data[g].get(model, 0) for g in groups]
        ax.barh(ys, vals, height=bar_h, color=MODEL_COLORS[model], label=model, zorder=3)
        for y, v in zip(ys, vals):
            ax.text(v + xlim[1] * 0.012, y, f"{v:.1f}%", va="center",
                     fontsize=9, color="#0b0b0b", fontweight="bold")

    ax.set_yticks(y_base)
    ax.set_yticklabels(groups, fontsize=11)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_xlim(0, xlim[1])
    _style_axes(ax)
    ax.legend(loc="lower right", frameon=False, fontsize=9, ncol=3, bbox_to_anchor=(1, -0.18))
    fig.suptitle(title, fontsize=13, fontweight="bold", x=0.02, ha="left", y=0.98)
    ax.set_title(subtitle, fontsize=9, color="#52514e", loc="left", pad=10)
    fig.tight_layout(rect=[0, 0.02, 1, 0.93])
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {out_path.name}")


def wins_chart(wins, ties, out_path):
    labels = ["Sonnet 4.6", "GPT-5.2", "GPT-4o", "Ties"]
    values = [wins["sonnet4_6"], wins["gpt5_2"], wins["gpt4o"], ties]
    colors = [MODEL_COLORS["Sonnet 4.6"], MODEL_COLORS["GPT-5.2"], MODEL_COLORS["GPT-4o"], "#b8b6ac"]
    total = sum(values)

    fig, ax = plt.subplots(figsize=(7, 3.2), dpi=200)
    bars = ax.barh(labels[::-1], values[::-1], color=colors[::-1], height=0.55, zorder=3)
    for b, v in zip(bars, values[::-1]):
        ax.text(v + 0.3, b.get_y() + b.get_height() / 2, f"{v} / {total}",
                 va="center", fontsize=10, fontweight="bold")
    ax.set_xlim(0, max(values) + 3)
    _style_axes(ax)
    ax.set_xlabel(f"Combinations won (out of {total} with gold data)", fontsize=10)
    fig.suptitle("Combo-Level Wins: Highest Exact-Match % per Combination",
                 fontsize=13, fontweight="bold", x=0.02, ha="left", y=0.98)
    ax.set_title("Each gold-covered combination counted once, regardless of gold-point volume",
                 fontsize=9, color="#52514e", loc="left", pad=10)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {out_path.name}")

## Main

In [ ]:
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    agg, by_behavior, by_student, per_combo = collect_comparisons()
    wins, ties = count_combo_wins(per_combo)

    total_per_model = {m: sum(agg[m].values()) for m in MODEL_TAGS}

    print(f"Combinations with gold data: {len(per_combo)}")
    for model_tag in MODEL_TAGS:
        total = total_per_model[model_tag]
        exact = agg[model_tag][1]
        miss = agg[model_tag][10]
        print(f"{MODEL_NAMES[model_tag]}: total={total}  "
              f"exact={exact} ({exact / total * 100:.1f}%)  "
              f"miss={miss} ({miss / total * 100:.1f}%)")
    print(f"Combo-level wins: {wins}  ties={ties}")

    # Chart 1: exact-match vs. complete-miss
    headline = {
        "Exact match rate": {MODEL_NAMES[m]: agg[m][1] / total_per_model[m] * 100 for m in MODEL_TAGS},
        "Complete miss rate": {MODEL_NAMES[m]: agg[m][10] / total_per_model[m] * 100 for m in MODEL_TAGS},
    }
    grouped_hbar_chart(
        headline,
        "Exact Match vs. Complete Miss (aggregated across all gold-covered combinations)",
        f"{sum(total_per_model.values()) // len(MODEL_TAGS)} gold-point comparisons per model",
        OUT_DIR / "01_headline_exact_miss.png", xlim=(0, 62),
    )

    # Chart 2: combo-level wins
    wins_chart(wins, ties, OUT_DIR / "02_combo_wins.png")

    # Chart 3: by behavior
    behavior_data = {
        beh: {MODEL_NAMES[m]: rate_pct(by_behavior[beh][m], "exact") for m in MODEL_TAGS if m in by_behavior.get(beh, {})}
        for beh in BEHAVIORS if beh in by_behavior
    }
    grouped_hbar_chart(
        behavior_data, "Exact-Match Rate by Behavior Type",
        "No single model wins every behavior category",
        OUT_DIR / "03_by_behavior.png", xlim=(0, 95),
    )

    # Chart 4: by student
    student_data = {
        stu: {MODEL_NAMES[m]: rate_pct(by_student[stu][m], "exact") for m in MODEL_TAGS if m in by_student.get(stu, {})}
        for stu in STUDENTS if stu in by_student
    }
    grouped_hbar_chart(
        student_data, "Exact-Match Rate by Student",
        "Differences track how much speech/gesture data each student's sessions contain",
        OUT_DIR / "04_by_student.png", xlim=(0, 75),
    )

    # Chart 5: full 10-category breakdown
    category_data = {
        CAT_LABELS[c]: {MODEL_NAMES[m]: agg[m][c] / total_per_model[m] * 100 for m in MODEL_TAGS}
        for c in range(1, 11)
    }
    grouped_hbar_chart(
        category_data, "Full 10-Category Boundary Classification (aggregated %)",
        "Category 1 = exact match, Category 10 = no overlap; 2-9 are partial matches",
        OUT_DIR / "05_full_category_breakdown.png", xlim=(0, 62),
    )


if __name__ == "__main__":
    main()